<a href="https://colab.research.google.com/github/madhukshara09/Codomax-Data-Science-Intern/blob/main/module2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# OBJECTIVE 1: Work with Datasets using Pandas and NumPy
# ==========================================
import numpy as np
import pandas as pd

print(f"Pandas Version: {pd.__version__} | NumPy Version: {np.__version__}\n")

# Create a raw dataset with intentional messiness (missing values, duplicates, unformatted text)
raw_data = {
    "Customer_ID": ["C101", "C102", "C103", "C104", "C105", "C106", "C107", "C108", "C102", "C109"],
    "Name": ["Aarav", "Diya", "Rohan", "Sneha", "Vikram", "Ananya", "Karthik", "Meera", "Diya", "Arjun"],
    "Plan_Type": ["pro", "Basic", "PRO", "basic", "Enterprise", "Pro", np.nan, "BASIC", "Basic", "Enterprise"],
    "Monthly_Spend": [120.50, 45.00, 89.99, np.nan, 250.00, 115.00, 50.00, np.nan, 45.00, 275.00],
    "Support_Tickets": [1, 5, 2, 6, 0, np.nan, 3, 4, 5, 1],
    "Tenure_Months": [24, 3, 14, 2, 36, 18, 6, 9, 3, 42]
}

# Save to a CSV file to simulate importing an external CSV dataset
raw_df = pd.DataFrame(raw_data)
raw_df.to_csv("raw_customer_telemetry.csv", index=False)


# ==========================================
# OBJECTIVE 2: Import CSV Datasets into Google Colab
# ==========================================
df = pd.read_csv("raw_customer_telemetry.csv")
print("--- RAW IMPORTED DATASET ---")
print(df)
print("\n")


# ==========================================
# OBJECTIVE 3: Explore, Clean, and Organize Data
# ==========================================
print("--- DATASET STRUCTURE & SUMMARY ---")
print(df.info())
print("\nMissing Values per Column:\n", df.isnull().sum())

# 1. Remove duplicate rows (C102 appears twice)
df_clean = df.drop_duplicates(subset=["Customer_ID"]).copy()

# 2. Standardize text formatting in 'Plan_Type' (e.g., 'pro', 'PRO' -> 'Pro')
df_clean["Plan_Type"] = df_clean["Plan_Type"].str.strip().str.title()


# ==========================================
# OBJECTIVE 4: Handle Missing Values & Apply Data Analysis Techniques
# ==========================================
# 1. Handle missing categorical values with the mode (most frequent plan)
mode_plan = df_clean["Plan_Type"].mode()[0]
df_clean["Plan_Type"] = df_clean["Plan_Type"].fillna(mode_plan)

# 2. Handle missing numerical values using median (spend) and 0 (tickets)
median_spend = df_clean["Monthly_Spend"].median()
df_clean["Monthly_Spend"] = df_clean["Monthly_Spend"].fillna(median_spend)
df_clean["Support_Tickets"] = df_clean["Support_Tickets"].fillna(0).astype(int)

# 3. Apply NumPy vectorized operations to create new analytical columns
df_clean["Annual_Value"] = np.round(df_clean["Monthly_Spend"] * 12, 2)
df_clean["Customer_Tier"] = np.where(df_clean["Annual_Value"] >= 1200, "High Value", "Standard")

# Sort dataset by Annual Value descending
df_clean = df_clean.sort_values(by="Annual_Value", ascending=False).reset_index(drop=True)

print("\n--- CLEANED & ORGANIZED DATASET ---")
print(df_clean)

# 4. Basic Data Analysis Techniques: Descriptive Stats, GroupBy, and Correlation
print("\n--- STATISTICAL SUMMARY (NUMERICAL) ---")
print(df_clean.describe().round(2))

print("\n--- REVENUE & TICKET ANALYSIS BY PLAN TYPE ---")
plan_summary = df_clean.groupby("Plan_Type").agg(
    Customer_Count=("Customer_ID", "count"),
    Avg_Monthly_Spend=("Monthly_Spend", "mean"),
    Total_Annual_Revenue=("Annual_Value", "sum"),
    Avg_Tickets=("Support_Tickets", "mean")
).round(2)
print(plan_summary)

# Correlation between Tenure and Monthly Spend using NumPy
corr_matrix = np.corrcoef(df_clean["Tenure_Months"], df_clean["Monthly_Spend"])
print(f"\nCorrelation between Tenure & Monthly Spend: {corr_matrix[0, 1]:.2f}")

# Export cleaned dataset
df_clean.to_csv("cleaned_customer_telemetry.csv", index=False)